1. Data Loading & Initial Understanding
Using Pandas for Data Exploration

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_style("whitegrid")
%matplotlib inline

# Load your data
df = pd.read_csv('your_dataset.csv')

# Initial exploration
print("="*50)
print("DATA OVERVIEW")
print("="*50)
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nFirst 5 rows:")
print(df.head())

print("\nColumn names and data types:")
print(df.dtypes)

print("\nBasic statistics:")
print(df.describe())

print("\nMissing values:")
print(df.isnull().sum())

2. Data Cleaning

2.1 Handle Duplicates

In [ ]:
# Check for duplicates
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")

# Remove duplicates if any
if duplicates > 0:
    df = df.drop_duplicates()
    print(f"Shape after removing duplicates: {df.shape}")

2.2 Check for Missing Values

In [ ]:
# Visualize missing values
plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.show()

# Handle missing values (choose one strategy per column)
# Option 1: Drop columns with too many missing values (>30%)
# df = df.drop(columns=['column_name'])

# Option 2: Fill with mean/median for numerical
# df['column'] = df['column'].fillna(df['column'].median())

# Option 3: Fill with mode for categorical
# df['column'] = df['column'].fillna(df['column'].mode()[0])

2.3 Detect and Handle Outliers

In [ ]:
# Create boxplots for numerical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns

# Plot multiple boxplots
fig, axes = plt.subplots(nrows=len(numerical_cols)//3 + 1, ncols=3, figsize=(15, 10))
axes = axes.ravel()

for i, col in enumerate(numerical_cols):
    if i < len(axes):
        axes[i].boxplot(df[col].dropna())
        axes[i].set_title(col)

# Hide empty subplots
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

# Remove outliers using IQR method
def remove_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

# Apply to numerical columns (be careful with target variable)
for col in numerical_cols:
    if col != 'target_column':  # Replace with actual target name
        original_shape = df.shape
        df = remove_outliers_iqr(df, col)
        print(f"Removed {original_shape[0] - df.shape[0]} outliers from {col}")

print(f"\nFinal shape after outlier removal: {df.shape}")

2.4 Convert Categorical Values

In [ ]:
# Identify categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns
print(f"Categorical columns: {list(categorical_cols)}")

# Convert to numerical using label encoding
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
for col in categorical_cols:
    df[col + '_encoded'] = le.fit_transform(df[col])
    print(f"Encoded {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Option: Drop original categorical columns if no longer needed
# df = df.drop(columns=categorical_cols)

3. Exploratory Data Analysis

3.1 Identify Target and Predictors

In [ ]:
# Define your target variable (domain knowledge required)
target = 'target_column_name'  # Replace with actual target
predictors = [col for col in df.columns if col != target and col not in categorical_cols]

print(f"Target variable: {target}")
print(f"Number of predictors: {len(predictors)}")
print(f"Predictors: {predictors[:5]}...")  # Show first 5

3.2 Univariate Analysis

In [ ]:
# Summary statistics
print("\n" + "="*50)
print("SUMMARY STATISTICS")
print("="*50)
print(df[predictors].describe())

# Distribution plots for key predictors
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(15, 10))
axes = axes.ravel()

for i, col in enumerate(predictors[:6]):  # Plot first 6 predictors
    axes[i].hist(df[col], bins=30, edgecolor='black', alpha=0.7)
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

3.3 Relationship with Target Variable

In [ ]:
# For regression problems (continuous target)
if df[target].dtype in ['float64', 'int64']:
    fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(15, 10))
    axes = axes.ravel()
    
    for i, col in enumerate(predictors[:6]):
        axes[i].scatter(df[col], df[target], alpha=0.5)
        axes[i].set_xlabel(col)
        axes[i].set_ylabel(target)
        axes[i].set_title(f'{target} vs {col}')
        
        # Add trend line
        z = np.polyfit(df[col], df[target], 1)
        p = np.poly1d(z)
        axes[i].plot(df[col].sort_values(), p(df[col].sort_values()), "r--", alpha=0.8)

# For classification problems (categorical target)
else:
    # Box plots for each predictor vs target
    fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(15, 10))
    axes = axes.ravel()
    
    for i, col in enumerate(predictors[:6]):
        df.boxplot(column=col, by=target, ax=axes[i])
        axes[i].set_title(f'{col} by {target}')

plt.tight_layout()
plt.show()

3.4 Multivariate Analysis - Correlation Heatmap

In [ ]:
# Calculate correlations
correlation_matrix = df[predictors + [target]].corr()

# Plot heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Heatmap', fontsize=16)
plt.tight_layout()
plt.show()

# Top correlations with target
target_correlations = correlation_matrix[target].sort_values(ascending=False)
print("\nTop features correlated with target:")
print(target_correlations)

3.5 Pairplot for Key Features

In [ ]:
# Select top 4-5 features most correlated with target
top_features = target_correlations.index[1:6]  # Skip target itself
if len(top_features) > 0:
    sns.pairplot(df[list(top_features) + [target]], diag_kind='kde')
    plt.suptitle('Pairplot of Top Features', y=1.02)
    plt.show()

4. Summary Report Template (Homework)

Create a markdown or Word document with the following sections:
# Exploratory Data Analysis Report
**Student Name:** 
**Class Number:** 
**Date:** 
**Dataset:** [Name and Source]

## 1. Dataset Overview
- Brief description of the dataset and its engineering relevance.
- Number of rows: 
- Number of columns: 
- Target variable: 
- Problem type (regression/classification): 
- If regression: explain whether class conversion was performed (if applicable).
- Summary statistics of numerical variables (mean, std, min, max).
 ## Required Evidence:
- Dataset shape (df.shape)
- Data types (df.dtypes)
- Summary statistics (df.describe())

## 2. Data Quality Assessment
- Missing values: [list columns with missing %]
- Duplicates found and handled: 
- Outliers detected and handled: 
- Data types after cleaning: 

## 3. Key Findings from Univariate Analysis
[Describe distributions of key variables - normal, skewed, etc.]
[Include 2-3 important plots with broef intrepretation]

## 4. Key Findings from Multivariate Analysis
- Top 3 features correlated with target:
  1. [Feature] - [Correlation coefficient]
  2. [Feature] - [Correlation coefficient]
  3. [Feature] - [Correlation coefficient]
- Interesting relationships between predictors:
[Describe any multicollinearity or patterns]
- Correlation Heatmap
  1. Include a correlation matrix heatmap.
  2. Identify multicollinearity issues (if any).
- Predictor Relationships
  1. Describe important relationships between predictors.
  2. Discuss patterns (linear, nonlinear, clusters, etc.).

## 5. Initial Insights for Modeling
- Which features seem most promising?
- What preprocessing might be needed?
  1. Scaling?
  2. Encoding?
  3. Feature selection?
- What modeling approaches might work well?

## 6. Engineering Interpretation
- What do the results mean in your discipline?
- What type of model might suit this dataset?

## 7. Challenges Encountered
[Describe any difficulties with data cleaning, outliers, etc.]

## 8. Reflection Questions
- How might poor data quality affect model reliability?
- What ethical or bias issues might exist in this dataset?

## 9. Next Steps
[What would you explore next with more time?]